# Shamir's (k, n) Threshold Secret Sharing Scheme

A secret value **S** is divided into **n shares**.

## Terminology

### Total Number of Participants (n)

The number of participants who receive shares.

### Threshold (k)

The secret can be reconstructed when **k or more shares** are collected.

### k − 1 or Fewer Participants

The secret cannot be reconstructed.

---

## Example

- n = 7
- k = 3

In this case:

- Shares are distributed to 7 participants.
- The secret can be reconstructed when 3 or more participants collaborate.
- The secret cannot be reconstructed with only 2 or fewer participants.

---

## Program Workflow

1. Set the secret **S**.
2. Generate a random polynomial.
3. Distribute shares to the participants.
4. Collect at least **k** shares.
5. Reconstruct the secret using **Lagrange interpolation**.

All computations are performed over the finite field **GF(p)**, so every arithmetic operation is carried out modulo **p**.

Because **xᵢ** values are assumed to be public, each participant's share consists only of **yᵢ**. Therefore, the size of each share is **log₂(p)** bits.

In [1]:
import random
#from functools import reduce

# =========================
# Prime modulus p defining the finite field
# =========================
# Shamir's Secret Sharing performs all computations
# over the finite field GF(p).
#
# p should be chosen as a prime number larger than
# both the secret value and the number of participants.
#
# Since the share size is proportional to log(p),
# choosing a very large p increases the size of each share.
#
# In this example, p = 101 is used.
PRIME = 101


# =========================
# Modular inverse
# =========================
# Computes a^(-1) mod p.
#
# In a finite field, division is performed by
# multiplying by the modular inverse rather than
# dividing directly.
#
def mod_inverse(a, p):
    return pow(a, -1, p)


# =========================
# Generate a random polynomial f(x)
# =========================
# f(x) = secret + a1*x + a2*x^2 + ...
#
# By placing the secret in the constant term,
#
# f(0) = secret
#
# holds.
#
# If k or more shares are collected,
# the polynomial can be reconstructed.
#
def generate_polynomial(secret, degree, prime):

    # Constant term = secret
    coeffs = [secret]

    # Generate the remaining coefficients randomly
    for _ in range(degree):
        coeffs.append(
            random.randrange(0, prime)
        )

    return coeffs


# =========================
# Evaluate polynomial f(x)
# =========================
# Receives:
#
# coeffs : list of polynomial coefficients
# x      : x-coordinate
#
# Example:
#
# coeffs = [a0, a1, a2]
#
# corresponds to
#
# f(x) = a0 + a1*x + a2*x^2
#
# The result is reduced modulo p so that
# it remains an element of GF(p).
#
def evaluate_polynomial(
    coeffs,
    x,
    prime
):

    y = 0

    # Add each term of the polynomial
    for power, coeff in enumerate(coeffs):

        y += (
            coeff *
            pow(x, power, prime)
        )

    return y % prime


# =========================
# Generate shares
# =========================
# Splits the secret into n shares.
#
# Each participant receives:
#
# (participant_id, x_i, y_i)
#
# where
#
# y_i = f(x_i)
#
# If at least k shares are collected,
# the polynomial can be reconstructed.
#
def split_secret(
    secret,
    n,
    k,
    prime=PRIME
):

    # The threshold cannot exceed
    # the number of participants
    if k > n:
        raise ValueError(
            "k must be less than or equal to n"
        )

    # Generate a polynomial of degree k - 1
    coeffs = generate_polynomial(
        secret,
        k - 1,
        prime
    )

    shares = []

    # -------------------------
    # Generate random x-values
    # -------------------------
    # Duplicate x-values must be avoided,
    # otherwise some share combinations
    # may become unusable for reconstruction.
    #
    # A set is used to ensure uniqueness.
    x_values = set()

    while len(x_values) < n:
        x_values.add(
            random.randrange(1, prime)
        )

    x_values = list(x_values)

    # -------------------------
    # Create shares
    # -------------------------
    for participant_id, x in enumerate(
        x_values,
        start=1
    ):

        # y = f(x)
        y = evaluate_polynomial(
            coeffs,
            x,
            prime
        )

        shares.append(
            (
                participant_id,
                x,
                y
            )
        )

    return shares, coeffs


# =========================
# Lagrange interpolation
# =========================
# Given points
#
# (x1, y1), (x2, y2), ...
#
# reconstructs the polynomial value
# at a specified point x.
#
# In Shamir's scheme, the secret is recovered
# by evaluating
#
# f(0)
#
# using the collected shares.
#
def lagrange_interpolation(
    x,
    x_s,
    y_s,
    prime
):

    total = 0

    # Number of shares used
    k = len(x_s)

    for i in range(k):

        xi = x_s[i]
        yi = y_s[i]

        numerator = 1
        denominator = 1

        for j in range(k):

            if i == j:
                continue

            xj = x_s[j]

            # Numerator
            numerator = (
                numerator *
                (x - xj)
            ) % prime

            # Denominator
            denominator = (
                denominator *
                (xi - xj)
            ) % prime

        # Lagrange basis polynomial
        lagrange_coeff = (
            numerator *
            mod_inverse(
                denominator,
                prime
            )
        )

        total += yi * lagrange_coeff

    return total % prime


# =========================
# Secret reconstruction
# =========================
# Recovers the secret from a set of shares.
#
# The secret is obtained by computing
#
# f(0)
#
# Since
#
# f(0) = secret
#
# the original secret is recovered.
#
def reconstruct_secret(
    shares,
    prime=PRIME
):

    # Extract x-coordinates
    x_s = [
        share[1]
        for share in shares
    ]

    # Extract y-coordinates
    y_s = [
        share[2]
        for share in shares
    ]

    # Evaluate the polynomial at x = 0
    return lagrange_interpolation(
        0,
        x_s,
        y_s,
        prime
    )

Example Execution

In [2]:
# =========================
# Example Execution
# =========================
if __name__ == "__main__":

    # -------------------------
    # Secret value S
    # -------------------------
    secret = 1

    # -------------------------
    # Total number of participants n
    # -------------------------
    n = 7

    # -------------------------
    # Threshold k
    # The secret can be reconstructed
    # when at least k shares are collected.
    # -------------------------
    k = 3

    print("==============================================")
    print("Shamir's (k, n) Threshold Secret Sharing Scheme")
    print("==============================================")

    print(f"Secret S                : {secret}")
    print(f"Total participants n    : {n}")
    print(f"Threshold k             : {k}")
    print(f"The secret can be reconstructed with at least {k} shares")
    print(f"The secret cannot be reconstructed with {k-1} or fewer shares")
    print("==============================================")

    # -------------------------
    # Split the secret into n shares
    # -------------------------
    shares, coeffs = split_secret(secret, n, k)

    print("\nGenerated Shares")
    print("(Participant ID, x-coordinate, y-coordinate)")
    print("----------------------------------------------")

    for s in shares:
        print(s)

    # -------------------------
    # Select shares to be used
    # for secret reconstruction
    # -------------------------
    reconstruction_count = 4

    selected_shares = random.sample(
        shares,
        reconstruction_count
    )

    print("\nShares Selected for Reconstruction")
    print("----------------------------------------------")

    for s in selected_shares:
        print(s)

    print(
        f"\nNumber of shares used: {reconstruction_count}"
    )

    # -------------------------
    # Reconstruct the secret
    # -------------------------
    recovered = reconstruct_secret(
        selected_shares
    )

    print("\nReconstruction Result")
    print("----------------------------------------------")
    print("Original secret      :", secret)
    print("Recovered secret     :", recovered)
    print("Match                :", secret == recovered)

Shamir's (k, n) Threshold Secret Sharing Scheme
Secret S                : 1
Total participants n    : 7
Threshold k             : 3
The secret can be reconstructed with at least 3 shares
The secret cannot be reconstructed with 2 or fewer shares

Generated Shares
(Participant ID, x-coordinate, y-coordinate)
----------------------------------------------
(1, 32, 26)
(2, 1, 95)
(3, 99, 33)
(4, 71, 72)
(5, 80, 19)
(6, 55, 42)
(7, 28, 53)

Shares Selected for Reconstruction
----------------------------------------------
(4, 71, 72)
(1, 32, 26)
(7, 28, 53)
(3, 99, 33)

Number of shares used: 4

Reconstruction Result
----------------------------------------------
Original secret      : 1
Recovered secret     : 1
Match                : True


## Adding New Participants

New shares can be issued to additional participants using the existing shares.

A share for a new participant is generated by:

- Selecting a new x-coordinate
- Computing the corresponding value \( y = f(x) \)

The secret **S** remains unchanged.

The shares already held by existing participants also remain unchanged.

### Maximum Number of Participants

In Shamir's Secret Sharing Scheme, each participant must be assigned a distinct x-coordinate in the finite field **GF(p)**.

Therefore, the number of participants is limited by the size of the finite field.

Since this implementation does not use **x = 0**, the theoretical maximum number of participants is **p − 1**.

If the number of participants reaches this limit, a larger finite field must be selected, and all shares must be regenerated and redistributed over the new field.

## Adding a New Participant (Without a Dealer)

A new share can be generated without a dealer by reconstructing the polynomial from **k existing shares** and evaluating it at a new x-coordinate.

Specifically:

1. Collect at least **k valid shares**.
2. Reconstruct the polynomial using Lagrange interpolation.
3. Choose a new x-coordinate that is not already in use.
4. Compute the corresponding value \( f(x) \).
5. Distribute the resulting share to the new participant.

As a result, a new participant can be added to the scheme.

The secret itself remains unchanged, and the shares held by existing participants can continue to be used without modification.

However, generating a new share requires collecting **at least k valid shares**, since the polynomial cannot be reconstructed with fewer than k shares.

In [3]:
# =========================
# Add New Participants Without a Dealer
# =========================
# Generates shares for new participants
# using existing shares, without requiring
# a trusted dealer.
#
# Inputs:
# shares               : existing shares
# start_participant_id : starting ID number
#                        for new participants
# add_n                : number of participants
#                        to add
#
# Output:
# List of newly generated shares
#
# This function computes f(x_new)
# using Lagrange interpolation.
#
# The secret remains unchanged.
# Existing shares remain valid and
# do not need to be modified.
#
def add_new_participants(
    shares,
    start_participant_id,
    add_n,
    prime=PRIME
):

    # Extract x-coordinates and y-coordinates
    # from the existing shares
    x_s = [share[1] for share in shares]
    y_s = [share[2] for share in shares]

    # Storage for newly generated shares
    new_shares = []

    # Keep track of x-values already in use
    # Each participant must have a unique
    # x-coordinate
    used_x = set(x_s)

    # Generate shares for the requested
    # number of new participants
    for i in range(add_n):

        # -------------------------
        # Generate a new x-coordinate
        # -------------------------
        while True:

            x_new = random.randrange(
                1,
                prime
            )

            # Select an unused x-coordinate
            if x_new not in used_x:

                used_x.add(x_new)
                break

        # -------------------------
        # Compute the new share
        # -------------------------
        # Evaluate f(x_new) using
        # Lagrange interpolation
        y_new = lagrange_interpolation(
            x_new,
            x_s,
            y_s,
            prime
        )

        # Create the new share
        # (participant_id, x, y)
        new_share = (
            start_participant_id + i,
            x_new,
            y_new
        )

        new_shares.append(
            new_share
        )

    # Return the newly generated shares
    return new_shares

## Example: Adding New Participants Without a Dealer

In [4]:
# -------------------------
# Number of new participants
# to be added
# -------------------------
add_n = 3


# -------------------------
# Existing shares used to
# generate new shares
# -------------------------
# Randomly select k shares.
#
# With k or more shares,
# the original polynomial f(x)
# can be reconstructed, allowing
# new shares to be generated.
#
# In this example, no dealer
# (trusted secret holder) is used.
# Only the shares held by
# participants are utilized.
#
base_shares = random.sample(
    shares,
    k
)


# -------------------------
# Generate new participants
# -------------------------
# Get the current number
# of participants.
#
# Example:
# If there are already
# 7 participants,
# new participants will be
# assigned IDs starting from 8.
#
current_n = len(shares)

new_shares = add_new_participants(
    base_shares,
    current_n + 1,
    add_n
)

print("\nNewly Added Participants:")

for s in new_shares:
    print(s)


# -------------------------
# Create the complete set
# of participants
# -------------------------
# Combine the original
# participants and the newly
# added participants.
#
all_shares = shares + new_shares

# To repeatedly add more
# participants in future rounds,
# uncomment the following line:
#
# shares = all_shares
#
# This allows the newly added
# participants to be included
# in subsequent participant sets.
#
# If left commented out,
# each addition starts from
# the original participant set.
#
# shares = all_shares

print("\nAll Participants:")

for s in all_shares:
    print(s)


# -------------------------
# Select participants for
# secret reconstruction
# -------------------------
# Randomly select k participants
# from the complete participant set.
#
# In Shamir's Secret Sharing Scheme,
# the secret can be reconstructed
# when at least k shares are available.
#
selected_shares = random.sample(
    all_shares,
    k
)

print("\nParticipants Used for Secret Reconstruction:")

for s in selected_shares:
    print(s)


# -------------------------
# Reconstruct the secret
# -------------------------
# Compute f(0) using
# Lagrange interpolation.
#
# In Shamir's Secret Sharing Scheme,
#
#     f(0) = Secret S
#
# Therefore, the original secret
# can be recovered.
#
recovered = reconstruct_secret(
    selected_shares
)

print("\nRecovered Secret:", recovered)

print("\nMatch:", recovered == secret)


Newly Added Participants:
(8, 64, 34)
(9, 18, 86)
(10, 94, 16)

All Participants:
(1, 32, 26)
(2, 1, 95)
(3, 99, 33)
(4, 71, 72)
(5, 80, 19)
(6, 55, 42)
(7, 28, 53)
(8, 64, 34)
(9, 18, 86)
(10, 94, 16)

Participants Used for Secret Reconstruction:
(9, 18, 86)
(7, 28, 53)
(1, 32, 26)

Recovered Secret: 1

Match: True


## Adding New Participants (With a Dealer)

If the dealer retains the coefficients of the polynomial \( f(x) \), new shares can be issued without collecting existing shares.

To generate a new share:

1. Select a new x-coordinate that is not already in use.
2. The dealer directly evaluates \( f(x) \) at that point.
3. The resulting value is distributed as a share to the new participant.

As a result, new participants can be added to the scheme without reconstructing the polynomial from existing shares.

The secret itself remains unchanged, and the shares held by existing participants continue to be valid without modification.

Unlike the dealerless approach, this method does not require collecting **k shares**. However, it requires the dealer to retain the polynomial information (i.e., the coefficients of \( f(x) \)) throughout the lifetime of the scheme.

In [5]:
# =========================
# Add New Participants With a Dealer
# =========================
#
# Inputs:
# coeffs               : polynomial coefficients
# existing_shares      : shares of existing participants
# start_participant_id : starting ID number
#                        for new participants
# add_n                : number of participants
#                        to add
#
# Output:
# List of newly generated shares
#
# In this approach, the dealer retains
# the polynomial f(x).
#
# Therefore, the dealer can directly
# compute f(x_new) without performing
# Lagrange interpolation.
#
# No existing shares need to be collected.
#
# The secret remains unchanged.
# Existing shares remain valid and
# do not need to be modified.
#
def add_new_participants_with_dealer(
    coeffs,
    existing_shares,
    start_participant_id,
    add_n,
    prime=PRIME
):

    # -------------------------
    # Obtain x-coordinates that
    # are already in use
    # -------------------------
    # Duplicate x-coordinates must
    # be avoided because they would
    # result in duplicate shares.
    used_x = {
        share[1]
        for share in existing_shares
    }

    # Storage for newly generated shares
    new_shares = []

    # -------------------------
    # Generate shares for the
    # requested number of
    # new participants
    # -------------------------
    for i in range(add_n):

        # -------------------------
        # Generate a new x-coordinate
        # -------------------------
        while True:

            x_new = random.randrange(
                1,
                prime
            )

            # Select an unused x-coordinate
            if x_new not in used_x:

                used_x.add(x_new)
                break

        # -------------------------
        # Compute the new share
        # -------------------------
        # Since the dealer retains
        # the polynomial f(x),
        #
        #     y_new = f(x_new)
        #
        # can be evaluated directly.
        y_new = evaluate_polynomial(
            coeffs,
            x_new,
            prime
        )

        # -------------------------
        # Create the share for the
        # new participant
        # -------------------------
        new_shares.append(
            (
                start_participant_id + i,
                x_new,
                y_new
            )
        )

    # Return the newly generated shares
    return new_shares

## Example: Adding New Participants With a Dealer

In [6]:
# =========================
# Example: Adding New Participants With a Dealer
# =========================

# -------------------------
# Number of new participants
# to be added
# -------------------------
add_n = 3


# -------------------------
# Generate new participants
# -------------------------
# The dealer retains the
# polynomial coefficients
# stored in coeffs.
#
# Therefore,
#
#     y = f(x)
#
# can be computed directly.
#
# Unlike the dealerless approach,
# there is no need to collect
# k shares and perform
# Lagrange interpolation.
#
# Get the current number
# of participants.
#
# Example:
# If there are already
# 7 participants,
# new participants will be
# assigned IDs starting from 8.
#
current_n = len(shares)

new_shares = add_new_participants_with_dealer(
    coeffs,
    shares,
    current_n + 1,
    add_n
)

print("\nNewly Added Participants:")

for s in new_shares:
    print(s)


# -------------------------
# Create the complete set
# of participants
# -------------------------
# Combine the original
# participants and the newly
# added participants.
#
all_shares = shares + new_shares

# To repeatedly add more
# participants in future rounds,
# uncomment the following line:
#
# shares = all_shares
#
# This allows the newly added
# participants to be included
# in subsequent participant sets.
#
# If left commented out,
# each addition starts from
# the original participant set.
#
# shares = all_shares

print("\nAll Participants:")

for s in all_shares:
    print(s)


# -------------------------
# Select participants for
# secret reconstruction
# -------------------------
# Randomly select k participants
# from the complete participant set.
#
# In Shamir's Secret Sharing Scheme,
# the secret can be reconstructed
# when at least k shares are available.
#
selected_shares = random.sample(
    all_shares,
    k
)

print("\nParticipants Used for Secret Reconstruction:")

for s in selected_shares:
    print(s)


# -------------------------
# Reconstruct the secret
# -------------------------
# Compute f(0) using
# Lagrange interpolation.
#
# In Shamir's Secret Sharing Scheme,
#
#     f(0) = Secret S
#
# Therefore, the original secret
# can be recovered.
#
recovered = reconstruct_secret(
    selected_shares
)

print("\nRecovered Secret:", recovered)

print("\nMatch:", recovered == secret)


Newly Added Participants:
(8, 47, 98)
(9, 93, 71)
(10, 45, 71)

All Participants:
(1, 32, 26)
(2, 1, 95)
(3, 99, 33)
(4, 71, 72)
(5, 80, 19)
(6, 55, 42)
(7, 28, 53)
(8, 47, 98)
(9, 93, 71)
(10, 45, 71)

Participants Used for Secret Reconstruction:
(2, 1, 95)
(4, 71, 72)
(9, 93, 71)

Recovered Secret: 1

Match: True
